In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

# Project directories
PROJECT_ROOT = "/content/drive/MyDrive/Projectwork/Betel_Leaf_Final"

STAGE5_ROOT = os.path.join(
    PROJECT_ROOT, "Stage5_Hybrid_Representation"
)

RESULTS_DIR = os.path.join(
    STAGE5_ROOT, "Stage5_1_Geometric_Features", "Results"
)

# File paths
GT_FULL_PATH = os.path.join(
    RESULTS_DIR, "stage5_1_geometric_features_gt_v2.csv"
)

GT_SELECTED_PATH = os.path.join(
    RESULTS_DIR, "stage5_1_geometric_features_selected_v1.csv"
)

PRED_FULL_PATH = os.path.join(
    RESULTS_DIR, "stage5_3_geometric_features_predicted_v1.csv"
)

PRED_SELECTED_PATH = os.path.join(
    RESULTS_DIR, "stage5_3_geometric_features_predicted_selected_v1.csv"
)

# Verify files exist
file_paths = {
    "GT full": GT_FULL_PATH,
    "GT selected": GT_SELECTED_PATH,
    "Predicted full": PRED_FULL_PATH,
    "Predicted selected": PRED_SELECTED_PATH
}

for name, path in file_paths.items():
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

# Load feature tables
gt_full = pd.read_csv(GT_FULL_PATH)
gt_selected = pd.read_csv(GT_SELECTED_PATH)

pred_full = pd.read_csv(PRED_FULL_PATH)
pred_selected = pd.read_csv(PRED_SELECTED_PATH)

# Display basic information
print("\nLOADED FEATURE TABLES")

for name, df in [
    ("Ground-truth full", gt_full),
    ("Ground-truth selected", gt_selected),
    ("Predicted full", pred_full),
    ("Predicted selected", pred_selected)
]:
    print(f"{name}: {df.shape}")

print("\nAll four tables loaded successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GT full: FOUND
GT selected: FOUND
Predicted full: FOUND
Predicted selected: FOUND

LOADED FEATURE TABLES
Ground-truth full: (720, 22)
Ground-truth selected: (720, 14)
Predicted full: (720, 22)
Predicted selected: (720, 14)

All four tables loaded successfully.


In [ ]:
# 1. Check duplicate filenames
tables = {
    "GT full": gt_full,
    "GT selected": gt_selected,
    "Predicted full": pred_full,
    "Predicted selected": pred_selected
}

print("DUPLICATE CHECK")

for name, df in tables.items():
    print(f"{name}: {df['filename'].duplicated().sum()} duplicate filenames")

# 2. Check whether all four tables contain the same filenames
print("\nFILENAME MATCHING")

reference_files = set(gt_full["filename"])

for name, df in tables.items():
    current_files = set(df["filename"])

    missing = reference_files - current_files
    extra = current_files - reference_files

    print(
        f"{name}: "
        f"missing={len(missing)}, "
        f"extra={len(extra)}, "
        f"total={len(current_files)}"
    )

# 3. Compare class and split assignments
print("\nCLASS AND SPLIT CONSISTENCY")

comparison = gt_full[
    ["filename", "class", "split"]
].merge(
    pred_full[["filename", "class", "split"]],
    on="filename",
    how="outer",
    suffixes=("_gt", "_pred"),
    indicator=True,
    validate="one_to_one"
)

matched = comparison[comparison["_merge"] == "both"].copy()

class_mismatches = (
    matched["class_gt"] != matched["class_pred"]
).sum()

split_mismatches = (
    matched["split_gt"] != matched["split_pred"]
).sum()

print("Matched records:", len(matched))
print("Unmatched records:", (comparison["_merge"] != "both").sum())
print("Class mismatches:", class_mismatches)
print("Split mismatches:", split_mismatches)

# 4. Display any inconsistencies
if class_mismatches > 0:
    print("\nClass mismatch examples:")
    print(
        matched.loc[
            matched["class_gt"] != matched["class_pred"],
            ["filename", "class_gt", "class_pred"]
        ].head()
    )

if split_mismatches > 0:
    print("\nSplit mismatch examples:")
    print(
        matched.loc[
            matched["split_gt"] != matched["split_pred"],
            ["filename", "split_gt", "split_pred"]
        ].head()
    )

# 5. Final status
print("\nVERIFICATION STATUS")

all_unique = all(
    not df["filename"].duplicated().any()
    for df in tables.values()
)

all_matching = all(
    set(df["filename"]) == reference_files
    for df in tables.values()
)

metadata_consistent = (
    len(matched) == len(gt_full)
    and class_mismatches == 0
    and split_mismatches == 0
)

if all_unique and all_matching and metadata_consistent:
    print("SUCCESS: All four tables are aligned and consistent.")
else:
    print("WARNING: Inconsistencies detected. Review the output before proceeding.")

DUPLICATE CHECK
GT full: 0 duplicate filenames
GT selected: 0 duplicate filenames
Predicted full: 0 duplicate filenames
Predicted selected: 0 duplicate filenames

FILENAME MATCHING
GT full: missing=0, extra=0, total=720
GT selected: missing=0, extra=0, total=720
Predicted full: missing=0, extra=0, total=720
Predicted selected: missing=0, extra=0, total=720

CLASS AND SPLIT CONSISTENCY
Matched records: 720
Unmatched records: 0
Class mismatches: 0
Split mismatches: 0

VERIFICATION STATUS
SUCCESS: All four tables are aligned and consistent.


In [ ]:
# STAGE 5.4 — BUILD GT vs PREDICTED COMPARISON


# Ten selected geometric features
selected_features = [
    "lesion_area_ratio_percent",
    "lesion_compactness",
    "lesion_component_count",
    "lesion_centroid_x",
    "lesion_centroid_y",
    "lesion_dispersion_x",
    "lesion_dispersion_y",
    "lesion_quadrant_1",
    "lesion_quadrant_2",
    "lesion_quadrant_3"
]

# Keep metadata and selected features
gt_comparison = gt_selected[
    ["filename", "class", "split", "empty_lesion_mask"] + selected_features
].copy()

pred_comparison = pred_selected[
    ["filename", "class", "split", "empty_lesion_mask"] + selected_features
].copy()

# Rename feature columns to distinguish ground truth and predictions
gt_comparison = gt_comparison.rename(
    columns={
        feature: f"{feature}_gt"
        for feature in selected_features
    }
).rename(
    columns={"empty_lesion_mask": "empty_lesion_mask_gt"}
)

pred_comparison = pred_comparison.rename(
    columns={
        feature: f"{feature}_pred"
        for feature in selected_features
    }
).rename(
    columns={"empty_lesion_mask": "empty_lesion_mask_pred"}
)

# Merge using filename
comparison_df = gt_comparison.merge(
    pred_comparison,
    on="filename",
    how="inner",
    suffixes=("_gt", "_pred"),
    validate="one_to_one"
)

# Verify metadata after merging
assert len(comparison_df) == 720
assert (comparison_df["class_gt"] == comparison_df["class_pred"]).all()
assert (comparison_df["split_gt"] == comparison_df["split_pred"]).all()

# Remove duplicate metadata columns and retain one copy
comparison_df = comparison_df.drop(
    columns=["class_pred", "split_pred"]
).rename(
    columns={
        "class_gt": "class",
        "split_gt": "split"
    }
)

print("COMPARISON TABLE")
print("Shape:", comparison_df.shape)
print("Total images:", len(comparison_df))

print("\nImages by split:")
print(comparison_df["split"].value_counts())

print("\nImages by class:")
print(comparison_df["class"].value_counts())

print("\nComparison columns:")
print(comparison_df.columns.tolist())

print("\nSample comparison:")
display(comparison_df.head())

print("\nSUCCESS: Ground-truth and predicted features are paired.")

COMPARISON TABLE
Shape: (720, 25)
Total images: 720

Images by split:
split
train         504
test          108
validation    108
Name: count, dtype: int64

Images by class:
class
healthy      247
bacterial    245
fungal       228
Name: count, dtype: int64

Comparison columns:
['filename', 'class', 'split', 'empty_lesion_mask_gt', 'lesion_area_ratio_percent_gt', 'lesion_compactness_gt', 'lesion_component_count_gt', 'lesion_centroid_x_gt', 'lesion_centroid_y_gt', 'lesion_dispersion_x_gt', 'lesion_dispersion_y_gt', 'lesion_quadrant_1_gt', 'lesion_quadrant_2_gt', 'lesion_quadrant_3_gt', 'empty_lesion_mask_pred', 'lesion_area_ratio_percent_pred', 'lesion_compactness_pred', 'lesion_component_count_pred', 'lesion_centroid_x_pred', 'lesion_centroid_y_pred', 'lesion_dispersion_x_pred', 'lesion_dispersion_y_pred', 'lesion_quadrant_1_pred', 'lesion_quadrant_2_pred', 'lesion_quadrant_3_pred']

Sample comparison:


,filename,class,split,empty_lesion_mask_gt,lesion_area_ratio_percent_gt,lesion_compactness_gt,lesion_component_count_gt,lesion_centroid_x_gt,lesion_centroid_y_gt,lesion_dispersion_x_gt,...,lesion_area_ratio_percent_pred,lesion_compactness_pred,lesion_component_count_pred,lesion_centroid_x_pred,lesion_centroid_y_pred,lesion_dispersion_x_pred,lesion_dispersion_y_pred,lesion_quadrant_1_pred,lesion_quadrant_2_pred,lesion_quadrant_3_pred
0,Bacterial_Leaf_Spot_Disease(105).jpg,bacterial,test,False,0.815686,0.111037,3,0.359171,0.648982,0.371516,...,0.827987,0.112205,4,0.337277,0.626431,0.369994,0.088253,0.097883,0.000000,0.624233
1,Bacterial_Leaf_Spot_Disease(106).jpg,bacterial,test,False,0.842114,0.161459,3,0.205478,0.718710,0.124777,...,0.842704,0.148017,4,0.243481,0.717969,0.121445,0.140709,0.084495,0.000000,0.858626
2,Bacterial_Leaf_Spot_Disease(111).jpg,bacterial,test,False,1.570916,0.094783,4,0.307947,0.685480,0.117465,...,1.455583,0.111465,5,0.297672,0.701325,0.113163,0.174285,0.015266,0.085550,0.897626
3,Bacterial_Leaf_Spot_Disease(114).jpg,bacterial,test,False,8.130496,0.073139,9,0.608671,0.441917,0.263600,...,8.178448,0.062676,16,0.609206,0.440806,0.258800,0.197943,0.113065,0.564842,0.274424
4,Bacterial_Leaf_Spot_Disease(118).jpg,bacterial,test,False,0.738872,0.160253,2,0.467039,0.779772,0.069245,...,0.738499,0.175762,3,0.465700,0.778350,0.069808,0.029474,0.000000,0.000000,0.491300



SUCCESS: Ground-truth and predicted features are paired.


In [ ]:
# STAGE 5.4 — TEST-SET FEATURE AGREEMENT

from scipy.stats import pearsonr, spearmanr

# Primary evaluation: held-out test set
test_df = comparison_df[
    comparison_df["split"] == "test"
].copy()

# Spatial features are evaluated only where GT lesions exist
diseased_test_df = test_df[
    test_df["empty_lesion_mask_gt"] == False
].copy()

# Features evaluated on all test images
all_image_features = [
    "lesion_area_ratio_percent",
    "lesion_compactness",
    "lesion_component_count"
]

# All ten features evaluated on diseased test images
diseased_features = selected_features.copy()


def calculate_feature_metrics(df, feature):
    gt = df[f"{feature}_gt"].astype(float).to_numpy()
    pred = df[f"{feature}_pred"].astype(float).to_numpy()

    errors = pred - gt

    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    bias = np.mean(errors)

    # Correlation is undefined for constant inputs
    if len(gt) >= 2 and np.std(gt) > 0 and np.std(pred) > 0:
        pearson = pearsonr(gt, pred).statistic
        spearman = spearmanr(gt, pred).statistic
    else:
        pearson = np.nan
        spearman = np.nan

    return {
        "feature": feature,
        "n": len(df),
        "MAE": mae,
        "RMSE": rmse,
        "Mean_Bias": bias,
        "Pearson_r": pearson,
        "Spearman_rho": spearman
    }


# 1. All test images: area, compactness, component count
all_results = []

for feature in all_image_features:
    all_results.append(
        calculate_feature_metrics(test_df, feature)
    )

all_metrics_df = pd.DataFrame(all_results)
all_metrics_df.insert(0, "evaluation_group", "All_test_images")


# 2. Diseased test images: all ten selected features
diseased_results = []

for feature in diseased_features:
    diseased_results.append(
        calculate_feature_metrics(diseased_test_df, feature)
    )

diseased_metrics_df = pd.DataFrame(diseased_results)
diseased_metrics_df.insert(0, "evaluation_group", "Diseased_test_images")


# Combine results
test_feature_metrics = pd.concat(
    [all_metrics_df, diseased_metrics_df],
    ignore_index=True
)

print("TEST-SET FEATURE AGREEMENT")
print("Total test images:", len(test_df))
print("Diseased test images:", len(diseased_test_df))
print("Healthy test images:", len(test_df) - len(diseased_test_df))

print("\nALL TEST IMAGES")
display(all_metrics_df.round(5))

print("\nDISEASED TEST IMAGES")
display(diseased_metrics_df.round(5))

print("\nEvaluation completed.")

TEST-SET FEATURE AGREEMENT
Total test images: 108
Diseased test images: 71
Healthy test images: 37

ALL TEST IMAGES


,evaluation_group,feature,n,MAE,RMSE,Mean_Bias,Pearson_r,Spearman_rho
0,All_test_images,lesion_area_ratio_percent,108,0.29205,0.95163,-0.21426,0.99081,0.98462
1,All_test_images,lesion_compactness,108,0.04957,0.11425,0.00930,0.77049,0.86967
2,All_test_images,lesion_component_count,108,0.72222,1.29815,0.40741,0.92743,0.92408



DISEASED TEST IMAGES


,evaluation_group,feature,n,MAE,RMSE,Mean_Bias,Pearson_r,Spearman_rho
0,Diseased_test_images,lesion_area_ratio_percent,71,0.43958,1.17318,-0.33058,0.98950,0.96496
1,Diseased_test_images,lesion_compactness,71,0.05883,0.10030,-0.00242,0.78189,0.81573
2,Diseased_test_images,lesion_component_count,71,1.07042,1.59223,0.59155,0.90035,0.75961
3,Diseased_test_images,lesion_centroid_x,71,0.02656,0.05464,-0.00729,0.96855,0.97710
4,Diseased_test_images,lesion_centroid_y,71,0.03647,0.08453,0.01958,0.89706,0.87616
5,Diseased_test_images,lesion_dispersion_x,71,0.02348,0.04993,0.00503,0.83949,0.78307
6,Diseased_test_images,lesion_dispersion_y,71,0.01783,0.04322,-0.00170,0.85575,0.85959
7,Diseased_test_images,lesion_quadrant_1,71,0.05434,0.16285,-0.00874,0.84346,0.83286
8,Diseased_test_images,lesion_quadrant_2,71,0.04279,0.12660,-0.02838,0.93177,0.92904
9,Diseased_test_images,lesion_quadrant_3,71,0.06995,0.18499,0.02618,0.84282,0.85672



Evaluation completed.


In [ ]:
# STAGE 5.4 — EMPTY LESION DETECTION

# Use held-out test images
empty_test = test_df.copy()

# Ground truth and predicted empty status
y_true = empty_test["empty_lesion_mask_gt"].astype(bool)
y_pred = empty_test["empty_lesion_mask_pred"].astype(bool)

# Confusion counts
TN = ((y_true == True) & (y_pred == True)).sum()
FP = ((y_true == True) & (y_pred == False)).sum()
FN = ((y_true == False) & (y_pred == True)).sum()
TP = ((y_true == False) & (y_pred == False)).sum()

# Metrics
accuracy = (TP + TN) / len(empty_test)
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0
)

print("EMPTY-LESION DETECTION")
print("Test images:", len(empty_test))

print("\nConfusion counts:")
print("Correctly detected non-empty lesions (TP):", TP)
print("Healthy images correctly detected as empty (TN):", TN)
print("Healthy images incorrectly predicted non-empty (FP):", FP)
print("Diseased images incorrectly predicted empty (FN):", FN)

print("\nMetrics:")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1-score:    {f1:.4f}")

print("\nCLASS-WISE EMPTY MASK CHECK")

for cls in ["bacterial", "fungal", "healthy"]:
    subset = empty_test[empty_test["class"] == cls]

    gt_empty = subset["empty_lesion_mask_gt"].astype(bool)
    pred_empty = subset["empty_lesion_mask_pred"].astype(bool)

    print(
        f"{cls}: total={len(subset)}, "
        f"GT empty={gt_empty.sum()}, "
        f"Predicted empty={pred_empty.sum()}, "
        f"GT empty but predicted non-empty={(gt_empty & ~pred_empty).sum()}, "
        f"GT non-empty but predicted empty={(~gt_empty & pred_empty).sum()}"
    )

EMPTY-LESION DETECTION
Test images: 108

Confusion counts:
Correctly detected non-empty lesions (TP): 71
Healthy images correctly detected as empty (TN): 35
Healthy images incorrectly predicted non-empty (FP): 2
Diseased images incorrectly predicted empty (FN): 0

Metrics:
Accuracy:    0.9815
Precision:   0.9726
Recall:      1.0000
Specificity: 0.9459
F1-score:    0.9861

CLASS-WISE EMPTY MASK CHECK
bacterial: total=37, GT empty=0, Predicted empty=0, GT empty but predicted non-empty=0, GT non-empty but predicted empty=0
fungal: total=34, GT empty=0, Predicted empty=0, GT empty but predicted non-empty=0, GT non-empty but predicted empty=0
healthy: total=37, GT empty=37, Predicted empty=35, GT empty but predicted non-empty=2, GT non-empty but predicted empty=0


In [ ]:
# STAGE 5.4 — SAVE EVALUATION RESULTS

import os
import json

# Output paths
feature_metrics_path = os.path.join(
    RESULTS_DIR,
    "stage5_4_test_feature_agreement_metrics_v1.csv"
)

empty_metrics_path = os.path.join(
    RESULTS_DIR,
    "stage5_4_empty_lesion_detection_metrics_v1.csv"
)

classwise_path = os.path.join(
    RESULTS_DIR,
    "stage5_4_classwise_empty_mask_results_v1.csv"
)

comparison_path = os.path.join(
    RESULTS_DIR,
    "stage5_4_gt_pred_feature_comparison_v1.csv"
)

summary_path = os.path.join(
    RESULTS_DIR,
    "stage5_4_evaluation_summary_v1.json"
)

# 1. Save feature agreement metrics
test_feature_metrics.to_csv(feature_metrics_path, index=False)

# 2. Save empty-lesion detection metrics
empty_metrics_df = pd.DataFrame([{
    "test_images": len(empty_test),
    "TP": int(TP),
    "TN": int(TN),
    "FP": int(FP),
    "FN": int(FN),
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "specificity": specificity,
    "f1_score": f1
}])

empty_metrics_df.to_csv(empty_metrics_path, index=False)

# 3. Save class-wise empty-mask results
classwise_rows = []

for cls in ["bacterial", "fungal", "healthy"]:
    subset = empty_test[empty_test["class"] == cls]

    gt_empty = subset["empty_lesion_mask_gt"].astype(bool)
    pred_empty = subset["empty_lesion_mask_pred"].astype(bool)

    classwise_rows.append({
        "class": cls,
        "total_images": len(subset),
        "GT_empty": int(gt_empty.sum()),
        "predicted_empty": int(pred_empty.sum()),
        "GT_empty_predicted_nonempty": int((gt_empty & ~pred_empty).sum()),
        "GT_nonempty_predicted_empty": int((~gt_empty & pred_empty).sum())
    })

classwise_df = pd.DataFrame(classwise_rows)
classwise_df.to_csv(classwise_path, index=False)

# 4. Save complete comparison table
comparison_df.to_csv(comparison_path, index=False)

# 5. Save a compact JSON summary
summary = {
    "evaluation": "Stage 5.4 - Predicted vs Ground-Truth Geometric Features",
    "test_images": int(len(test_df)),
    "diseased_test_images": int(len(diseased_test_df)),
    "healthy_test_images": int(len(test_df) - len(diseased_test_df)),
    "empty_lesion_detection": {
        "TP": int(TP),
        "TN": int(TN),
        "FP": int(FP),
        "FN": int(FN),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1_score": float(f1)
    }
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

# 6. Verify saved files
saved_files = [
    feature_metrics_path,
    empty_metrics_path,
    classwise_path,
    comparison_path,
    summary_path
]

print("STAGE 5.4 SAVE VERIFICATION")

for path in saved_files:
    print(
        os.path.basename(path),
        "—",
        "SAVED" if os.path.exists(path) else "NOT FOUND"
    )

print("\nStage 5.4 evaluation results saved.")

STAGE 5.4 SAVE VERIFICATION
stage5_4_test_feature_agreement_metrics_v1.csv — SAVED
stage5_4_empty_lesion_detection_metrics_v1.csv — SAVED
stage5_4_classwise_empty_mask_results_v1.csv — SAVED
stage5_4_gt_pred_feature_comparison_v1.csv — SAVED
stage5_4_evaluation_summary_v1.json — SAVED

Stage 5.4 evaluation results saved.


In [ ]:
# STAGE 5.5 — CREATE GEOMETRIC FEATURE MODULE

import os

SCRIPT_DIR = os.path.join(
    STAGE5_ROOT,
    "Stage5_1_Geometric_Features",
    "Scripts"
)

os.makedirs(SCRIPT_DIR, exist_ok=True)

MODULE_PATH = os.path.join(
    SCRIPT_DIR,
    "stage5_geometric_features.py"
)

module_code = r'''
import numpy as np
import pandas as pd

from PIL import Image
from scipy import ndimage
from skimage.measure import perimeter


def extract_geometric_features(
    row,
    leaf_mask_col="leaf_mask_path",
    lesion_mask_col="lesion_mask_path"
):
    """
    Extract geometric and spatial lesion features from leaf and lesion masks.

    The same function supports ground-truth and predicted masks by allowing
    the caller to specify the corresponding mask-path columns.

    Lesion pixels are not clipped to the leaf boundary.
    """

    leaf_mask = np.array(
        Image.open(row[leaf_mask_col]).convert("L")
    ) > 0

    lesion_mask = np.array(
        Image.open(row[lesion_mask_col]).convert("L")
    ) > 0

    if leaf_mask.shape != lesion_mask.shape:
        raise ValueError(
            f"Mask dimension mismatch for {row['filename']}: "
            f"leaf={leaf_mask.shape}, lesion={lesion_mask.shape}"
        )

    leaf_pixels = int(leaf_mask.sum())
    lesion_pixels = int(lesion_mask.sum())

    outside_leaf_pixels = int(
        (lesion_mask & ~leaf_mask).sum()
    )

    lesion_area_ratio = (
        lesion_pixels / leaf_pixels * 100
        if leaf_pixels > 0 else 0.0
    )

    empty_lesion = lesion_pixels == 0

    # Default values for empty lesion masks
    lesion_perimeter = 0.0
    normalized_perimeter = 0.0
    compactness = 0.0
    component_count = 0
    largest_component_ratio = 0.0
    mean_component_area = 0.0

    centroid_x = 0.0
    centroid_y = 0.0
    dispersion_x = 0.0
    dispersion_y = 0.0

    quadrant_1 = 0.0
    quadrant_2 = 0.0
    quadrant_3 = 0.0
    quadrant_4 = 0.0

    if not empty_lesion:

        # Perimeter and compactness
        lesion_perimeter = float(
            perimeter(lesion_mask, neighborhood=8)
        )

        if leaf_pixels > 0:
            normalized_perimeter = (
                lesion_perimeter / np.sqrt(leaf_pixels)
            )

        if lesion_perimeter > 0:
            compactness = (
                4 * np.pi * lesion_pixels /
                (lesion_perimeter ** 2)
            )

        # 8-connected lesion components
        structure = np.ones((3, 3), dtype=np.uint8)

        labeled_mask, component_count = ndimage.label(
            lesion_mask,
            structure=structure
        )

        component_areas = np.bincount(
            labeled_mask.ravel()
        )[1:]

        if len(component_areas) > 0:
            largest_component_ratio = (
                component_areas.max() / lesion_pixels
            )

            mean_component_area = float(
                component_areas.mean()
            )

        # Leaf bounding box
        leaf_y, leaf_x = np.where(leaf_mask)

        if len(leaf_x) > 0:

            x_min, x_max = leaf_x.min(), leaf_x.max()
            y_min, y_max = leaf_y.min(), leaf_y.max()

            bbox_width = max(x_max - x_min + 1, 1)
            bbox_height = max(y_max - y_min + 1, 1)

            # Lesion coordinates
            lesion_y, lesion_x = np.where(lesion_mask)

            # Normalized centroid
            centroid_x = (
                (lesion_x.mean() - x_min) / bbox_width
            )

            centroid_y = (
                (lesion_y.mean() - y_min) / bbox_height
            )

            # Normalized spatial dispersion
            dispersion_x = (
                lesion_x.std() / bbox_width
            )

            dispersion_y = (
                lesion_y.std() / bbox_height
            )

            # Quadrant distribution
            midpoint_x = x_min + bbox_width / 2
            midpoint_y = y_min + bbox_height / 2

            top = lesion_y < midpoint_y
            left = lesion_x < midpoint_x

            quadrant_1 = np.sum(top & left) / lesion_pixels
            quadrant_2 = np.sum(top & ~left) / lesion_pixels
            quadrant_3 = np.sum(~top & left) / lesion_pixels
            quadrant_4 = np.sum(~top & ~left) / lesion_pixels

    return {
        "filename": row["filename"],
        "class": row["class"],
        "split": row["split"],
        "leaf_pixels": leaf_pixels,
        "lesion_pixels": lesion_pixels,
        "lesion_pixels_outside_leaf": outside_leaf_pixels,
        "lesion_area_ratio_percent": lesion_area_ratio,
        "lesion_perimeter_pixels": lesion_perimeter,
        "normalized_lesion_perimeter": normalized_perimeter,
        "lesion_compactness": compactness,
        "lesion_component_count": int(component_count),
        "largest_component_ratio": largest_component_ratio,
        "mean_component_area_pixels": mean_component_area,
        "lesion_centroid_x": centroid_x,
        "lesion_centroid_y": centroid_y,
        "lesion_dispersion_x": dispersion_x,
        "lesion_dispersion_y": dispersion_y,
        "lesion_quadrant_1": quadrant_1,
        "lesion_quadrant_2": quadrant_2,
        "lesion_quadrant_3": quadrant_3,
        "lesion_quadrant_4": quadrant_4,
        "empty_lesion_mask": empty_lesion
    }


def extract_features_from_manifest(
    manifest,
    leaf_mask_col="leaf_mask_path",
    lesion_mask_col="lesion_mask_path"
):
    """Extract geometric features for every row in a manifest."""

    records = []

    for _, row in manifest.iterrows():
        records.append(
            extract_geometric_features(
                row,
                leaf_mask_col=leaf_mask_col,
                lesion_mask_col=lesion_mask_col
            )
        )

    return pd.DataFrame(records)
'''

with open(MODULE_PATH, "w") as f:
    f.write(module_code)

print("Module created:", os.path.exists(MODULE_PATH))
print("Module path:", MODULE_PATH)
print("File size:", os.path.getsize(MODULE_PATH), "bytes")

Module created: True
Module path: /content/drive/MyDrive/Projectwork/Betel_Leaf_Final/Stage5_Hybrid_Representation/Stage5_1_Geometric_Features/Scripts/stage5_geometric_features.py
File size: 5455 bytes


In [ ]:
# Load the Stage 5.2 predicted-mask inference manifest

INFERENCE_MANIFEST_PATH = os.path.join(
    RESULTS_DIR,
    "stage5_2_predicted_mask_inference_manifest_v1.csv"
)

manifest = pd.read_csv(INFERENCE_MANIFEST_PATH)

print("Manifest loaded:", manifest.shape)

print("\nAvailable columns:")
print(manifest.columns.tolist())

print("\nFirst record:")
display(manifest.head(1))

Manifest loaded: (720, 10)

Available columns:
['filename', 'class', 'split', 'original_width', 'original_height', 'predicted_leaf_pixels', 'predicted_lesion_pixels', 'predicted_lesion_outside_leaf_pixels', 'leaf_mask_path', 'lesion_mask_path']

First record:


,filename,class,split,original_width,original_height,predicted_leaf_pixels,predicted_lesion_pixels,predicted_lesion_outside_leaf_pixels,leaf_mask_path,lesion_mask_path
0,Bacterial_Leaf_Spot_Disease(105).jpg,bacterial,test,3000,3000,5784390,47894,0,/content/drive/MyDrive/Projectwork/Betel_Leaf_...,/content/drive/MyDrive/Projectwork/Betel_Leaf_...


In [ ]:
# ============================================
# STAGE 5.5 — MODULE REPRODUCIBILITY TEST
# ============================================

from stage5_geometric_features import extract_geometric_features

# Select one test image
sample_row = manifest.iloc[0]

# Extract features using the reusable module
module_result = extract_geometric_features(
    sample_row,
    leaf_mask_col="leaf_mask_path",
    lesion_mask_col="lesion_mask_path"
)

# Load the previously saved predicted features
saved_sample = pred_full[
    pred_full["filename"] == sample_row["filename"]
].iloc[0]

# Compare all numeric feature outputs
numeric_features = [
    col for col in pred_full.columns
    if col not in ["filename", "class", "split", "empty_lesion_mask"]
]

print("========== REPRODUCIBILITY TEST ==========")
print("Image:", sample_row["filename"])

for feature in numeric_features:
    new_value = float(module_result[feature])
    saved_value = float(saved_sample[feature])

    difference = abs(new_value - saved_value)

    print(
        f"{feature}: "
        f"saved={saved_value:.8f}, "
        f"module={new_value:.8f}, "
        f"difference={difference:.10f}"
    )

print("\nEmpty lesion status:")
print("Saved:", saved_sample["empty_lesion_mask"])
print("Module:", module_result["empty_lesion_mask"])

========== REPRODUCIBILITY TEST ==========
Image: Bacterial_Leaf_Spot_Disease(105).jpg
leaf_pixels: saved=5784390.00000000, module=5784390.00000000, difference=0.0000000000
lesion_pixels: saved=47894.00000000, module=47894.00000000, difference=0.0000000000
lesion_pixels_outside_leaf: saved=0.00000000, module=0.00000000, difference=0.0000000000
lesion_area_ratio_percent: saved=0.82798705, module=0.82798705, difference=0.0000000000
lesion_perimeter_pixels: saved=2316.00000000, module=2316.00000000, difference=0.0000000000
normalized_lesion_perimeter: saved=0.96296338, module=0.96296338, difference=0.0000000000
lesion_compactness: saved=0.11220543, module=0.11220543, difference=0.0000000000
lesion_component_count: saved=4.00000000, module=4.00000000, difference=0.0000000000
largest_component_ratio: saved=0.37075625, module=0.37075625, difference=0.0000000000
mean_component_area_pixels: saved=11973.50000000, module=11973.50000000, difference=0.0000000000
lesion_centroid_x: saved=0.33727736

In [ ]:
# STAGE 5.5 — FULL DATASET REPRODUCIBILITY

from stage5_geometric_features import extract_features_from_manifest

print("Starting feature extraction for 720 images...")

# Re-extract features using the reusable module
module_features = extract_features_from_manifest(
    manifest,
    leaf_mask_col="leaf_mask_path",
    lesion_mask_col="lesion_mask_path"
)

print("\nExtraction completed.")
print("Module feature table shape:", module_features.shape)

# Compare metadata
metadata_match = (
    module_features["filename"].equals(pred_full["filename"])
    and module_features["class"].equals(pred_full["class"])
    and module_features["split"].equals(pred_full["split"])
)

# Compare all numeric features
numeric_features = [
    col for col in pred_full.columns
    if col not in ["filename", "class", "split", "empty_lesion_mask"]
]

comparison_results = []

for feature in numeric_features:
    differences = (
        module_features[feature].astype(float)
        - pred_full[feature].astype(float)
    ).abs()

    comparison_results.append({
        "feature": feature,
        "max_absolute_difference": differences.max(),
        "mean_absolute_difference": differences.mean(),
        "exact_match": bool((differences == 0).all()),
        "within_tolerance": bool(
            np.allclose(
                module_features[feature].astype(float),
                pred_full[feature].astype(float),
                atol=1e-12,
                rtol=0
            )
        )
    })

reproducibility_df = pd.DataFrame(comparison_results)

# Compare empty-mask status
empty_status_match = (
    module_features["empty_lesion_mask"].astype(bool).equals(
        pred_full["empty_lesion_mask"].astype(bool)
    )
)

print("\nREPRODUCIBILITY RESULTS")
print("Total images:", len(module_features))
print("Metadata match:", metadata_match)
print("Empty-mask status match:", empty_status_match)

display(reproducibility_df)

all_features_match = reproducibility_df["within_tolerance"].all()

if metadata_match and empty_status_match and all_features_match:
    print("\nSUCCESS: All 720 images reproduce the saved features.")
else:
    print("\nWARNING: Differences detected. Review the comparison table.")

Starting feature extraction for 720 images...

Extraction completed.
Module feature table shape: (720, 22)

REPRODUCIBILITY RESULTS
Total images: 720
Metadata match: True
Empty-mask status match: True


,feature,max_absolute_difference,mean_absolute_difference,exact_match,within_tolerance
0,leaf_pixels,0.000000e+00,0.000000e+00,True,True
1,lesion_pixels,0.000000e+00,0.000000e+00,True,True
2,lesion_pixels_outside_leaf,0.000000e+00,0.000000e+00,True,True
3,lesion_area_ratio_percent,7.105427e-15,8.594772e-17,False,True
4,lesion_perimeter_pixels,0.000000e+00,0.000000e+00,True,True
5,normalized_lesion_perimeter,4.440892e-16,2.907107e-17,False,True
6,lesion_compactness,9.714451e-17,2.006979e-17,False,True
7,lesion_component_count,0.000000e+00,0.000000e+00,True,True
8,largest_component_ratio,1.110223e-16,1.175757e-17,False,True
9,mean_component_area_pixels,2.910383e-11,3.366394e-13,False,False


In [ ]:
# FINAL REPRODUCIBILITY CONFIRMATION

all_features_match = all(
    np.allclose(
        module_features[col].astype(float),
        pred_full[col].astype(float),
        atol=1e-9,
        rtol=0
    )
    for col in numeric_features
)

empty_status_match = (
    module_features["empty_lesion_mask"].astype(bool).equals(
        pred_full["empty_lesion_mask"].astype(bool)
    )
)

metadata_match = (
    module_features["filename"].equals(pred_full["filename"])
    and module_features["class"].equals(pred_full["class"])
    and module_features["split"].equals(pred_full["split"])
)

print("Numeric features match:", all_features_match)
print("Empty-mask status matches:", empty_status_match)
print("Metadata matches:", metadata_match)

if all_features_match and empty_status_match and metadata_match:
    print("\nSUCCESS: Module reproducibility verified for all 720 images.")
else:
    print("\nWARNING: Further investigation required.")

Numeric features match: True
Empty-mask status matches: True
Metadata matches: True

SUCCESS: Module reproducibility verified for all 720 images.
